# Evolution of Sonar & Sonar Signal Processing Fundamentals
Instructors: Spencer J. Chang (PhD), Daniel D. Sternlicht (PhD)

**Notebook Description:**
This notebook covers the basic concepts and exercises for processing sonar signals after they have been digitized with an A/D converter.

**Distribution A:** Approved for public release. Distribution is unlimited.

Content is largely based on standard reference materials.

In [ ]:
#### Python Imports across the whole notebook ####
import numpy as np
import scipy
import matplotlib.pyplot as plt

# It would be informative to skim the attributes
#   for the stave data to know what is getting loaded.
from ReadStaveData import StaveData
from joblib import Parallel, delayed

## Signal Characteristics

If we want to do analog signal processing, we must use *analog* components, like resistors, capacitors, inductors, transistors, and operational amplifiers in circuits that modify received signals. If we want to do **digital signal processing** as we are in this tutorial, we must first receive the signal through an analog-to-digital converter.

This carries with it a series of implications. One of the biggest implications that affects us is the chosen sampling rate.

**What do we mean by sampling rate?**

When collecting the amplitude information of incoming signals, we're restricted by our hardware (e.g., how many different amplitude levels we can receive) and *by how often we record the amplitude value of the signal.*

Note: `numpy.sinc` computes the *normalized* `sinc` function $\frac{sin(\pi x)}{\pi x}$, so to get a signal up to 15 Hz, $x=2(15)t$.

In [ ]:
# -------------------------------------------------
# 1. Define a time axis (e.g., 0‑2 seconds, 1 kHz sampling)
# -------------------------------------------------
fs = 20                # Sampling frequency [Hz]
t_end = 2                # Duration [seconds]
t = np.arange(0, t_end, 1/fs)

# -------------------------------------------------
# 2. Generate (normalized) sinc function signals
# -------------------------------------------------
impulse_f = 20
sig_a = lambda t_val: scipy.signal.hilbert(1.0 * np.sinc(2*impulse_f*(t_val-0.412)))
sig_b = lambda t_val: scipy.signal.hilbert(1.0 * np.sinc(2*impulse_f*(t_val-0.637)))
sig_c = lambda t_val: scipy.signal.hilbert(1.0 * np.sinc(2*impulse_f*(t_val-0.8)))
sig_d = lambda t_val: scipy.signal.hilbert(1.0 * np.sinc(2*impulse_f*(t_val-1)))

# -------------------------------------------------
# 3. Plot everything for visual verification
# -------------------------------------------------
sig_analysis = sig_a   # Change this to look at (potential) differences in FFT output

plt.figure(figsize=(10, 5))

# Plot each component
plt.subplot(2, 1, 1)
plt.plot(t, sig_a(t).real)
plt.plot(t, sig_b(t).real)
plt.plot(t, sig_c(t).real)
plt.plot(t, sig_d(t).real)
plt.title('Individual Sinusoidal Components')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')
plt.grid(True)

n_fft = len(t)
sig_fft = scipy.fft.fft(sig_analysis(t), n=n_fft)
freq_axis = scipy.fft.fftfreq(n=n_fft, d=1/fs)

plt.subplot(2, 1, 2)
plt.plot(scipy.fft.fftshift(freq_axis),
         scipy.fft.fftshift(np.abs(sig_fft)))
plt.title("Analyzed Signal: Fourier Domain")
plt.xlabel("Frequency")
plt.ylabel("Magnitude")

tw_ax = plt.twinx(plt.gca())
tw_ax.plot(scipy.fft.fftshift(freq_axis),
         scipy.fft.fftshift(np.angle(sig_fft)), c='tab:orange')
tw_ax.set_ylabel("Phase")
plt.tight_layout();

### Aliasing

In [ ]:
# Example of aliasing when undersampling a signal.
fs = 60                # Sampling frequency [Hz]
t_end = 2                # Duration [seconds]
t = np.arange(0, t_end, 1/fs)

s_freq = 20
c_freq = 50
sine_cos = lambda t_val: np.sin(2*np.pi*s_freq*(t_val-0.412)) + np.cos(2*np.pi*c_freq*t_val)

plt.figure(figsize=(10, 5))

# Plot each component
plt.subplot(2, 1, 1)
plt.plot(t, sine_cos(t))
plt.title('Composed Signal')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')
plt.grid(True)

n_fft = len(t)
sig_fft = scipy.fft.fft(sine_cos(t), n=n_fft)
freq_axis = scipy.fft.fftfreq(n=n_fft, d=1/fs)

plt.subplot(2, 1, 2)
plt.plot(scipy.fft.fftshift(freq_axis),
         scipy.fft.fftshift(np.abs(sig_fft)))
plt.title("Frequency Domain")
plt.xlabel("Frequency")
plt.ylabel("Magnitude")
plt.tight_layout();

Notice how the 50 Hz sinusoid is show up as approximately 7 Hz instead. When we undersample the input data, aliasing of the signal is often what happens. Essentially, the sinusoid's original 50 Hz frequency becomes "known as" (aliased) a 10 Hz signal.

### Passband and Baseband
Other terms that serve similar purposes:
- Carrier frequency (passband)
- Base frequency (baseband)
- Complex envelope (baseband)
- (Carrier) Modulated signal (passband)
- Modulating signal (baseband)
- Intermediate/Physical/Real signal (passband)

Below is a practical example of how to shift frequencies up and down in software.

In [ ]:
# Example: Frequency shift of signal in time-/frequency-domain.
s_freq = 20
freq_axis = scipy.fft.fftfreq(n=len(t), d=1/fs)
sig_data = np.sin(2*np.pi*s_freq*t)
sig_data = scipy.signal.hilbert(sig_data)

## Time-domain frequency-shifting with Euler's equation
shift_down = sig_data * np.exp(-2j * np.pi * 4 * t)
shift_up = sig_data * np.exp(2j * np.pi * 4 * t)
sig_fft = scipy.fft.fft(sig_data)
down_fft = scipy.fft.fft(shift_down)
up_fft = scipy.fft.fft(shift_up)

## Frequency-domain frequency-shifting (uses frequency sampling)
shift_hz = 3
bin_spacing = fs / len(t)
bins_to_shift = int(round(shift_hz / bin_spacing))
down_fft_2 = np.roll(down_fft, -bins_to_shift)

## Time-domain signals
down_time = np.fft.ifft(down_fft)
up_time = np.fft.ifft(up_fft)
down_time_2 = np.fft.ifft(down_fft_2)

plt.figure(figsize=(7, 3))
plt.plot(scipy.fft.fftshift(freq_axis), scipy.fft.fftshift(np.abs(sig_fft)), label="Original")
plt.plot(scipy.fft.fftshift(freq_axis), scipy.fft.fftshift(np.abs(down_fft)), label="Down")
plt.plot(scipy.fft.fftshift(freq_axis), scipy.fft.fftshift(np.abs(up_fft)), label="Up")
plt.plot(scipy.fft.fftshift(freq_axis), scipy.fft.fftshift(np.abs(down_fft_2)), label="Down2")
plt.xlabel("Frequency")
plt.ylabel('Magnitude')
plt.legend();

plt.figure(figsize=(9, 3))
plt.plot(t, sig_data.real, label="Original")
plt.plot(t, up_time.real, "*--", label="Up")
plt.xlabel("Time (s)")
plt.ylabel('Signal')
plt.xlim(0, 0.5)
plt.legend();

plt.figure(figsize=(9, 3))
plt.plot(t, sig_data.real, label="Original")
plt.plot(t, down_time.real, "o--", label="Down")
plt.plot(t, down_time_2.real, ".--", label="Down2")
plt.xlabel("Time (s)")
plt.ylabel('Signal')
plt.xlim(0, 0.5)
plt.legend();

### Phase Information
Example: Determining time-of-arrival for a received impulse signal. Notice that the magnitude in the frequency domain is always the same, but the phase information changes.

In [ ]:
fs = 60
t_end = 2                # Duration [seconds]
t = np.arange(0, t_end, 1/fs)

sig_close = np.pad([1], (0, len(t)-1))
sig_mid = np.pad([1], (10, len(t)-11))  # 90 Hz
sig_far = np.pad([1], (100, len(t)-101))

# Lowered sampling frequency at basebanded signals
fft_close = scipy.fft.fft(sig_close, n=len(t))
fft_mid = scipy.fft.fft(sig_mid, n=len(t))
fft_far = scipy.fft.fft(sig_far, n=len(t))
freq_axis = scipy.fft.fftfreq(n=len(t), d=1/fs)

plt.figure(figsize=(8, 6))

# Plot each component
plt.subplot(3, 2, 1)
plt.plot(t, sig_close)
plt.title('Close Impulse')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')
plt.subplot(3, 2, 3)
plt.plot(t, sig_mid)
plt.title('Mid-Range Impulse')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')
plt.subplot(3, 2, 5)
plt.plot(t, sig_far)
plt.title('Far-Range Impulse')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')

plt.subplot(3, 2, 2)
plt.plot(scipy.fft.fftshift(freq_axis), scipy.fft.fftshift(np.abs(fft_close)))
plt.plot(scipy.fft.fftshift(freq_axis), scipy.fft.fftshift(np.angle(fft_close)))
plt.subplot(3, 2, 4)
plt.plot(scipy.fft.fftshift(freq_axis), scipy.fft.fftshift(np.abs(fft_mid)))
plt.plot(scipy.fft.fftshift(freq_axis), scipy.fft.fftshift(np.angle(fft_mid)))
plt.subplot(3, 2, 6)
plt.plot(scipy.fft.fftshift(freq_axis), scipy.fft.fftshift(np.abs(fft_far)))
plt.plot(scipy.fft.fftshift(freq_axis), scipy.fft.fftshift(np.angle(fft_far)))
plt.xlabel("Frequency")
plt.tight_layout();

In [ ]:
# Short example: Hilbert Transform to get our "analytic signal"
sig_data = sine_cos(t)
sig_data = scipy.signal.hilbert(sig_data)
plt.figure(figsize=(7, 3))
plt.plot(sig_data.real, label="real")
plt.plot(sig_data.imag, label="imag")
plt.legend();

# Create figure and 3D axis
high_res_t = np.linspace(0, t_end, 500*t_end)
high_res_sig = scipy.signal.hilbert(sine_cos(high_res_t))
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot(high_res_t, high_res_sig.real, high_res_sig.imag)
ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('In', fontweight='bold')
ax.set_zlabel('Quad', fontweight='bold')
# ax.set_xlim(0, 0.25)
ax.view_init(elev=0, azim=-0);

## Fourier Analysis

In the frequency domain, it can be easier to do **filtering, downsampling, frequency shifting, and time delaying**.
We are only showing the downsampling and frequency shifting in this section.

- Citation for data in `manatee.npy`: *Rycyk, A., V. Cargille, D. Bolaji, C. Factheu, U. Ejimadu, C. Berchem, and A. Takoukam Kamla. 2025. Bioacoustic Dataset of African and Florida Manatee Vocalizations for Machine Learning Applications, 2020-2022 ver 2. Environmental Data Initiative. https://doi.org/10.6073/pasta/c73edcb4a36ed07aebfbe238a31ceb19*.
- The original sampling frequency was 44.1kHz.

In [ ]:
og_samples = np.load("manatee.npy")
fs = 44100
t_axis = np.arange(0, len(og_samples), step=1) / fs

# Normalize the waveform to range [-1, 1]
og_samples = (og_samples - og_samples.mean()) / (np.ptp(og_samples) / 2)   # Take out the DC

n_fft = len(og_samples)
audio_fft = scipy.fft.fft(og_samples, n=n_fft)
freq_axis = scipy.fft.fftfreq(n_fft, d=1/fs)

# Print summary
print(f"Audio duration: {len(og_samples)/fs:.2f} seconds")
print(f"Sample rate: {fs} Hz ({len(og_samples)} samples)")

plt.figure(figsize=(8, 3))
plt.plot(t_axis, og_samples.real)
plt.xlabel("Time (s)")
plt.ylabel("Normalized Acoustic Signal");

plt.figure(figsize=(8, 3))
plt.plot(scipy.fft.fftshift(freq_axis) / 1000,
         np.abs(scipy.fft.fftshift(audio_fft)))
plt.xlabel("Frequency (kHz)")
plt.ylabel("Magnitude");

### Downsampling in the frequency domain

To upsample, you could simply pad with zeros at the center of the Fourier transform array (not the `fftshift` version).

In [ ]:
new_fs = fs // 2
new_nfft = n_fft // 2   # 22050 Hz

# Apply downsampling in frequency domain
ds_fft = np.concatenate((audio_fft[:new_nfft//2], audio_fft[-new_nfft//2:]))
new_freq_axis = scipy.fft.fftfreq(new_nfft, d=1/new_fs)

# Get the downsampled audio signal
ds_audio = scipy.fft.ifft(ds_fft)
new_t_axis = np.arange(0, len(ds_audio), step=1) / new_fs

plt.figure(figsize=(8,3))
plt.plot(scipy.fft.fftshift(new_freq_axis) / 1000,
         np.abs(scipy.fft.fftshift(ds_fft)))
plt.xlabel("Frequency (kHz)")
plt.ylabel("Magnitude")

plt.figure(figsize=(8,3))
plt.plot(t_axis, og_samples.real, "o-", label="Original")
plt.plot(new_t_axis, ds_audio.real, "o--", label="Downsampled")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend(fontsize=14)
plt.xlim(-1e-5, 0.001)
plt.ylim(-0.6, 0.8);

### Frequency shifting in the frequency domain

This is the same as that shown above in the previous section "Signal Characteristics." Notice that we get some artifacts when shifting up (and aren't careful with how values get rolled around the array).

In [ ]:
f_shift = 1500
bin_spacing = new_fs / len(freq_axis)
bins_to_shift = int(round(f_shift / bin_spacing))
shifted_fft = np.roll(ds_fft, bins_to_shift)
dwnshift_audio = scipy.fft.ifft(shifted_fft)

# Create visualization (spectrogram)
shift_f_spec, shift_t_spec, shift_audio_spec = scipy.signal.spectrogram(dwnshift_audio, fs=new_fs, nperseg=128, nfft=256)
shift_f_spec = np.round(shift_f_spec/1000, decimals=2)
shift_t_spec = np.round(shift_t_spec, decimals=2)

plt.figure(figsize=(4, 8))
plt.imshow(shift_audio_spec)
ax = plt.gca()
x_idx = np.asarray(ax.get_xticks(), dtype=np.uint)[1:-1]
y_idx = np.asarray(ax.get_yticks(), dtype=np.uint)[1:-1]

plt.xticks(x_idx, shift_t_spec[x_idx] / 2, rotation=45)
plt.yticks(y_idx, shift_f_spec[y_idx])
plt.xlabel("Time (s)")
plt.ylabel("Frequency (kHz)")
plt.xlim(0, len(shift_t_spec)//2);
plt.ylim(len(shift_f_spec)//2, 0);

# Display overall frequency domain
plt.figure(figsize=(8,3))
plt.plot(scipy.fft.fftshift(new_freq_axis) / 1000,
         np.abs(scipy.fft.fftshift(ds_fft)), alpha=0.75, label="Original")
plt.plot(scipy.fft.fftshift(new_freq_axis) / 1000,
         np.abs(scipy.fft.fftshift(shifted_fft)), label="Shifted")
plt.xlabel("Frequency (kHz)")
plt.ylabel("Magnitude")
plt.legend(fontsize=14);

## Matched Filtering

Filtering typically involves the removing of undesired parts of an input signal while preserving the desired parts of an input signal.

In [ ]:
recv_data = np.load("sas_recv_pings.npy")   # Only taking one point along-track
proj_signal = np.load("sas_proj_signal.npy")

In [ ]:
compressed_data = np.load("sas_pulse_compress.npy")

In [ ]:
# Files to use: "sas_recv_pings.npy" and "sas_proj_signal.npy"
fs = 240e3
# TODO - Write code to perform pulse compression across the time-axis for every returned ping


In [ ]:
# TODO - Plot one row of range-compressed data alongside the raw time-series to prove that your algorithm works.


## Delay-and-Sum Beamforming: Linear Receiver Array

Let's consider what happens when a sound wave hits a two-element receiver array. Each of the sound wave's peaks are represented by the lines below, so we will call this a "plane wave" going forward (ie. a "plane" of sound hits each receiver).

<center>
    <img src="recv_array.png" width="600">
</center>

What we need to solve for is the time delay for the plane wave hitting the second (bottom) receiver. Notice that the distance from the initial plane wave to the second receiver is denoted by $\ell$. Allowing transmission loss between elements to be negligible, we will define our time series equations for the receivers:

\begin{align*}
    r_0(t) &= A e^{j\omega t}\\
    r_1(t) &= A e^{j\omega (t - \tau)}
\end{align*}

How will we solve for $\tau$? The answer is that we'll use $\ell = c \tau$ to help us figure out the answer. Recall that $c$ is the speed of sound underwater and could change slightly depending on how deep the receiver is.

We'll skip $\tau$'s mostly straightforward derivation, but to see the details beforehand, feel free to expand the details below.

<details>

<summary><b>Mathematical Derivation</b></summary>

\begin{align*}
    \ell = c \tau,&\ \ell = d \sin\theta\\
    \implies c \tau &= d \sin\theta\\
    \tau &= \frac{d}{c} \sin\theta
\end{align*}

</details>

Ultimately, if we expand the notation and break apart the complex exponentials for $r_1(t)$ above, we get
$$r_1(t) = A e^{j\omega t} e^{-j 2\pi f\frac{d}{c} \sin\theta} = A e^{j\omega t} e^{-j 2\pi \frac{d}{\lambda} \sin\theta}.$$

What happens when we include multiple receivers in the array and ensure that all have equal inter-element spacing? We can see that the receiver equations will be as follows:

\begin{align*}
    r_0(t) &= A e^{j\omega t}\\
    r_1(t) &= A e^{j\omega t} e^{-j 2\pi \frac{d}{\lambda} \sin\theta}\\
    r_2(t) &= A e^{j\omega t} e^{-j 2\pi \frac{2d}{\lambda} \sin\theta}\\
    \vdots\\
    r_{N-1}(t) &= A e^{j\omega t} e^{-j 2\pi \frac{(N-1)d}{\lambda} \sin\theta}
\end{align*}

We can put this into vector form: $\mathbf{y} = [r_0(t_0)\ r_1(t_0)\ \dots \ r_{N-1}(t_0)]$. This way,

\begin{align*}
    \mathbf{y} &= \hat{A} [1\ e^{-j k_{\theta}}\ \dots \ e^{-j 2\pi (N-1)k_{\theta}}]\\
               &= \hat{A} \mathbf{a}_s(\theta)\\
\end{align*}
where $k_{\theta} = 2\pi \frac{d}{\lambda} \sin\theta$. We will call $\mathbf{a}_s(\theta)$ the "steering vector" pointed toward the angle $\theta$.

To focus/steer our array's 'beam', we should therefore compute the dot product $\mathbf{y} \cdot \mathbf{a}^*_s(\theta)^T$. This will return a scaled version of whatever was received at the angle $\theta$ (relative to the linear array).

In [ ]:
def w_mvdr(theta, X, d_cumsum):
    # TODO - Implement this function for the MVDR beamforming method
    pass

In [ ]:
def est_music(theta, X, d_cumsum, En=None, num_signals=1):
    """
    Computes the MUSIC pseudo-spectrum weight for a given angle theta.

    Args:
        theta (float): Angle of arrival in radians.
        X (np.ndarray): Sonar array data, shape (N_elements, M_samples).
        d_cumsum (np.ndarray): Element-wise distances from the first element
                               divided by the wavelength, shape (N_elements,).
        En (np.ndarray): list of eigenvector arrays for the signal X. If None,
                         calculate the list. O/w, use En.
        num_signals (int): Expected number of noise signals (default is 1).

    Returns:
        float: The MUSIC pseudo-spectrum value for the given angle.
    """
    M = X.shape[1]

    eigenvalues = None
    if En is None:
        R = (X @ X.conj().T) / M

        eigenvalues, eigenvectors = np.linalg.eigh(R)

        # Sort eigenvalues and eigenvectors in descending order
        idx = np.argsort(np.abs(eigenvalues))[::-1]
        eigenvectors = eigenvectors[:, idx]
        eigenvalues = eigenvalues[idx]
        En = eigenvectors[:, num_signals:]

    a_theta = np.exp(-2j * np.pi * d_cumsum * np.sin(theta))
    a_theta = a_theta.reshape(-1, 1) # Format as column vector

    # 5. Compute the MUSIC pseudo-spectrum weight
    # Weight = 1 / (a^H * En * En^H * a)
    projection = a_theta.conj().T @ En @ En.conj().T @ a_theta

    # The result of the projection is theoretically real; extract the real part of the scalar
    music_pow = np.abs(1.0 / (projection.squeeze() + 1e-12))

    return music_pow, eigenvalues

In [ ]:
Nr = 5 # 5 elements
d = 0.5  # inter-element distance div'd by wavelength
d_dist = d * np.arange(Nr)

sample_rate = 1e6   # 1 MHz sampling
N = 10000 # number of samples to simulate
t = np.arange(N) / sample_rate # time vector

# Create tones to act as the transmitter signal
theta1 = 20 / 180 * np.pi # convert to radians
theta2 = 40 / 180 * np.pi
theta3 = -20 / 180 * np.pi

s1 = np.exp(-2j * np.pi * d_dist * np.sin(theta1)).reshape(-1,1) # Nr x 1
s2 = np.exp(-2j * np.pi * d_dist * np.sin(theta2)).reshape(-1,1)
s3 = np.exp(-2j * np.pi * d_dist * np.sin(theta3)).reshape(-1,1)

# we'll use 3 different frequencies.  1xN
tone1 = scipy.signal.hilbert(np.cos(2*np.pi*0.01e6*t)).reshape(1,-1)
tone2 = scipy.signal.hilbert(np.cos(2*np.pi*0.02e6*t)).reshape(1,-1)
tone3 = scipy.signal.hilbert(np.cos(2*np.pi*0.03e6*t)).reshape(1,-1)

X = s1 @ tone1 + 2*s2 @ tone2 + 0.1 * s3 @ tone3 # note the last one is 1/10th the power
n = scipy.signal.hilbert(np.random.randn(Nr, N)) #+ 1j*np.random.randn(Nr, N)
X = X + 0.1*n # 8xN

In [ ]:
theta_scan = np.linspace(-0.5*np.pi, 0.5*np.pi, 500) # 500 different thetas between -90 and +90 degrees
results = []
results_das = []
results_music = []
n_exp_sigs = 3      # MUSIC: Number of eigenvectors for noise subspace

for theta_i in theta_scan:
   # TODO - Implement w_das code and w_mvdr(...)
   w_das = None # Conventional, aka delay-and-sum, beamformer
   w_mv = w_mvdr(theta_i, X, d_cumsum=d_dist) # 3x1
   X_weighted = None   # TODO - Apply MVDR weights
   X_das = None        # TODO - Apply DAS weights
   var_music, _ = est_music(theta_i, X, d * np.arange(X.shape[0]), num_signals=n_exp_sigs)

   power_dB = 10*np.log10(np.var(X_weighted)) # power in signal, in dB so its easier to see small and large lobes at the same time
   power_das_dB = 10*np.log10(np.var(X_das)) # power in signal, in dB so its easier to see small and large lobes at the same time
   music_out = 10*np.log10(var_music)

   results.append(power_dB)
   results_das.append(power_das_dB)
   results_music.append(music_out)

results -= np.max(results) # normalize
results_music -= np.max(results_music) # normalize
results_das -= np.max(results_das) # normalize


### DAS vs MVDR

In [ ]:
#### DELAY-AND-SUM BEAMFORMING PLOT ###
theta_max = theta_scan[np.argmax(results_das)]
# Plot all results
fig, ax = plt.subplots(1, 2, subplot_kw={'projection': 'polar'}, figsize=(10, 4))
ax[0].set_title("DAS Beamforming")
ax[0].plot(theta_scan, results_das) # MAKE SURE TO USE RADIAN FOR POLAR
ax[0].plot([theta_max], [np.max(results_das)],'ro')
ax[0].text(theta_max - 0.1, np.max(results_das) - 4, np.round(theta_max * 180 / np.pi))
ax[0].set_theta_zero_location('N') # make 0 degrees point up
ax[0].set_theta_direction(-1) # increase clockwise
ax[0].set_rlabel_position(55)  # Move grid labels away from other labels
ax[0].set_thetamin(-90) # only show top half
ax[0].set_thetamax(90)

#### MINIMUM VARIANCE DISTORTIONLESS RESPONSE PLOT ###
theta_max = theta_scan[np.argmax(results)]
ax[1].set_title("MVDR Beamforming")
ax[1].plot(theta_scan, results) # MAKE SURE TO USE RADIAN FOR POLAR
ax[1].plot([theta_max], [np.max(results)],'ro')
ax[1].text(theta_max - 0.1, np.max(results) - 4, np.round(theta_max * 180 / np.pi))
ax[1].set_theta_zero_location('N') # make 0 degrees point up
ax[1].set_theta_direction(-1) # increase clockwise
ax[1].set_rlabel_position(55)  # Move grid labels away from other labels
ax[1].set_thetamin(-90) # only show top half
ax[1].set_thetamax(90)
plt.show()

### MVDR vs MUSIC

In [ ]:
#### MUltiple SIgnal Classification BEAMFORMING PLOT ###
theta_max = theta_scan[np.argmax(results_music)]
# Plot all results
fig, ax = plt.subplots(1, 2, subplot_kw={'projection': 'polar'}, figsize=(10, 4))
ax[0].set_title("MUSIC Beamforming")
ax[0].plot(theta_scan, results_music) # MAKE SURE TO USE RADIAN FOR POLAR
ax[0].plot([theta_max], [np.max(results_music)],'ro')
ax[0].text(theta_max - 0.1, np.max(results_music) - 4, np.round(theta_max * 180 / np.pi))
ax[0].set_theta_zero_location('N') # make 0 degrees point up
ax[0].set_theta_direction(-1) # increase clockwise
ax[0].set_rlabel_position(55)  # Move grid labels away from other labels
ax[0].set_thetamin(-90) # only show top half
ax[0].set_thetamax(90)

#### MINIMUM VARIANCE DISTORTIONLESS RESPONSE PLOT ###
theta_max = theta_scan[np.argmax(results)]
ax[1].set_title("MVDR Beamforming")
ax[1].plot(theta_scan, results) # MAKE SURE TO USE RADIAN FOR POLAR
ax[1].plot([theta_max], [np.max(results)],'ro')
ax[1].text(theta_max - 0.1, np.max(results) - 4, np.round(theta_max * 180 / np.pi))
ax[1].set_theta_zero_location('N') # make 0 degrees point up
ax[1].set_theta_direction(-1) # increase clockwise
ax[1].set_rlabel_position(55)  # Move grid labels away from other labels
ax[1].set_thetamin(-90) # only show top half
ax[1].set_thetamax(90)
plt.show()

## Synthetic Aperture Sonar Beamforming

The basic math synthetic aperture sonar (SAS) beamforming imagery is straightforward.
However, the most difficult part of the process is the bookkeeping. In other words, your algorithm must keep all the variables in order and computed correctly for an image to be processed into a recognizable image (i.e., it looks like the seafloor, an object of interest).

### Na&iuml;ve Approach - TD-Backprojection
What we'll discuss briefly in this tutorial is the method of **backprojection**. The one-sentence explanation is that we compute the exact (expected) time-delay at *each* pixel in our ranging grid for *each* of our transducer elements at *each* of the platform's positions such that it may be inverted to align the ping data at said platform positions.

The five steps we can split this method into are:
1. Point Range
1. Time Delay
1. Invert Time Delay
1. Beamforming Summation
1. Beamformed Magnitude

In pseudocode, it goes something like the following:
1. Pulse compression of raw, complex signal data (using known projected signal).
1. If necessary, perform basebanding, filtering, and downsampling in the fast-time direction (cross-track).
1. ***For each platform*** position,....
    1. ***For each transducer*** in the array,...
        1. Compute the FFT of the transducer's time series.
        1. ***For each pixel*** in the along-track and cross-track grid,...
            1. Compute the time-delay
            1. Shift the transducer's frequency response by the time-delay amount across all frequencies.
            1. Add the mean across the tranducer's frequency response.
            1. Store the mean in the pixel's variable.

Using [MASTODON](https://github.com/Sonar-Sim/MASTODON) (PR, v0.22.6) to simulate ping data, let's make our own SAS image beamformer with TD-Backprojection.

In [ ]:
def pulse_compression(recv_signal:np.ndarray, pulse:np.ndarray):
    """
    Perform pulse compression on a given receiver's signal.

    recv_signal may be 2D; time series is expected on last axis (eg. -1)
    pulse is assumed to be 1D and a single projected signal.

    TODO - Fill in the blank with your matched filtering algorithm.
    """
    pass

In [ ]:
# Create a grid of ranging coordinates related to along-track and cross-track position.
def range_grid_calc(src_grid:np.ndarray, tgt_grid:np.ndarray, axis=-1):
    assert src_grid.shape == tgt_grid.shape, f"Improper shapes between src and tgt: {src_grid.shape, tgt_grid.shape}"
    return np.linalg.vector_norm(src_grid - tgt_grid, axis=axis, ord=2)

# For each ping, compute the time delay and add the time-delayed series to a storage array the same size as the grid of ranging coord's.
# - Remember to apply the time delay across all possible frequencies for the ping's time series.
def process_single_ping(k, n_ele, stave_c, recv_pos, ele_loc, tgt_grid,
                        compressed_stave, freq_pts, arr_sum:int=-1,
                        verbose=0):
    # TODO - Compute the TD-backprojection for a single ping (along-track)
    pass

In [ ]:
# Load data from the sonar pings
stave_data = StaveData("Pointlinear1.h5")

In [ ]:
# Some variables to make things easier later
do_bband = True
n_cpu_jobs = 8
use_n_ele = -1   # Set to -1 or > stave_data.n_ele to choose all possible elements in transducer array
batch_sz = 1
decimate_factor = stave_data.fs / stave_data.bw
print(decimate_factor)

print_stave_idx = 0   # Whether to print platform, stave, element indices when beamforming

In [ ]:
# Pulse Compression of input time series
# Both signal and projected data are in passband from MASTODON
compressed_stave = pulse_compression(stave_data.data, stave_data.signal.signal)

In [ ]:
# Baseband the data. One advantage is that computation is easier.
if do_bband:
    print("[Basebanding = yes]")

    bb_timesamp = np.arange(stave_data.n_time) / stave_data.fs
    bb_shift = np.exp(-2j * np.pi * stave_data.fc * bb_timesamp)[np.newaxis, :]
    compressed_stave = compressed_stave * bb_shift

    ### METHOD 1 - Lowpass to clean out shifted data, then decimate. ###
    butter_b, butter_a = scipy.signal.butter(3, stave_data.bw, btype='lowpass', fs=stave_data.fs)
    compressed_stave = scipy.signal.filtfilt(butter_b, butter_a, compressed_stave)

    ### METHOD 2 - Fourier Domain; chop off 'zeroed' frequency components ###
    temp_f_ax = scipy.fft.fftfreq(stave_data.n_time, d=1/stave_data.fs)
    bw_idx = np.where(temp_f_ax == stave_data.bw/2)[0][0]   # bw/2 because I already know where to cut off

    bb_fft = scipy.fft.fft(compressed_stave, axis=1)
    bb_fft = np.hstack((bb_fft[:, :bw_idx], bb_fft[:, -bw_idx:]))
    compressed_stave = scipy.fft.ifft(bb_fft)

    print(f"[Signal Decimation by {decimate_factor} --> {compressed_stave.shape[-1]} time samples]")
    dec_fs = stave_data.fs / decimate_factor
else:
    print("[Basebanding = no]")
    dec_fs = stave_data.fs

In [ ]:
r_ax = np.linspace(0, stave_data.r_max, compressed_stave.shape[-1])  # Range coord's
xr_ax = np.linspace(0, stave_data.dy * (stave_data.n_pings-1), stave_data.n_pings) # X-Range coord's (Edit: stave_data.recv.pos.time[-1])
xr_ax = xr_ax - xr_ax.max()/2
z_motion = np.ones_like(xr_ax) * stave_data.motion[0]['z']

###### Get grid points for all "target locations" ######
stave_data.recv.positions = np.concatenate(   # NOTE - X-Y swapped because bearing=-90-deg
    (stave_data.recv.positions[1:2, :],
        stave_data.recv.positions[0:1, :],
        stave_data.recv.positions[2:3, :]), axis=0
)
platform_loc = np.hstack((np.zeros_like(xr_ax)[: ,np.newaxis], xr_ax[:,np.newaxis], z_motion[:, np.newaxis]))

#### SET 'TARGET' LOCATIONS ####
grid_xr = np.linspace(0, stave_data.dy * (stave_data.n_pings-1), stave_data.n_pings)
grid_xr = grid_xr - grid_xr.max()/2
r_pts, xr_pts = np.meshgrid(r_ax, grid_xr)
# Range, X-Range, Depth
tgt_loc_grid = np.concatenate((r_pts[..., np.newaxis],
                                xr_pts[..., np.newaxis],
                                10*np.ones_like(r_pts)[..., np.newaxis]), axis=-1)

## Compute each element location's specific range grid to get proportional change in time delay.
n_pts = stave_data.n_pings
total_n_pts = stave_data.n_pings
td_bf_sum = np.zeros(tgt_loc_grid.shape[:-1], dtype=np.complex64)
f_axis = scipy.fft.fftfreq(compressed_stave.shape[-1], d=1/dec_fs) + stave_data.fc
print(f"[Freq. Range --> [{f_axis.min():e}, {f_axis.max():e}]\n")

print(f"***** TD-beamforming for {n_pts} of {total_n_pts} points ({use_n_ele} elements) *****\n")
results = Parallel(n_jobs=n_cpu_jobs, batch_size=batch_sz, verbose=10)(
    delayed(process_single_ping)(
        k,
        stave_data.n_ele,
        stave_data.c,
        platform_loc,
        stave_data.recv.positions,
        tgt_loc_grid,
        compressed_stave,
        f_axis,
        arr_sum=use_n_ele,
        verbose=print_stave_idx
    ) for k in range(0, total_n_pts)  #, stave_data.n_ele)
)
# Add up all responses...
td_bf_sum = np.abs(np.sum(results, axis=0))**2
td_bf_sum = td_bf_sum / (td_bf_sum.max() + 1e-12)

In [ ]:
"""
This code plots...
    1) the pulse-compressed ping data in an image,
    2) the beamformed image
    3) the along-track view at range with greatest SNR
    4) the cross-track view at the middle of the along-track location
"""
cmap_choice = "bone"  # "pink", "bone"

# Crop the padded TD-beamforming array
print(f"***** Post-processing and Figure Plotting *****\n")

# Get maximum response; plot outputs
max_match = np.argmax(np.abs(compressed_stave).max(axis=0))
max_resp_idx = np.argmax(td_bf_sum.max(axis=0))
plt_r_axis = np.round(np.linspace(stave_data.r_min, stave_data.r_max,
                                    num=compressed_stave.shape[-1]), decimals=1)
plt_trk_axis = np.round(xr_ax, decimals=1)

fig, axes = plt.subplots(4, 1, figsize=(10, 9), layout='constrained')
axes[0].imshow(np.abs(compressed_stave),
                extent=[plt_r_axis.min(), plt_r_axis.max(), plt_trk_axis.min(), plt_trk_axis.max()],
                cmap=cmap_choice)
axes[0].set_title("Max Matched Filter Response")
axes[0].set_xlabel("Cross-Track (m)")
axes[0].set_ylabel("Along-Track (m)")
ax_lim = max(0, max_match-150), min(len(plt_r_axis), max_match+150)
axes[0].set_xlim(plt_r_axis[ax_lim[0]], plt_r_axis[ax_lim[1]])

axes[1].imshow(10 * np.log(td_bf_sum),
                extent=[plt_r_axis.min(), plt_r_axis.max(), plt_trk_axis.min(), plt_trk_axis.max()],
                cmap=cmap_choice)
axes[1].set_title("Beamforming at Max Response")
axes[1].set_xlabel("Cross-Track (m)")
axes[1].set_ylabel("Along-Track (m)")
ax_lim = max(0, max_match-150), min(len(plt_r_axis), max_match+150)
axes[1].set_xlim(plt_r_axis[ax_lim[0]], plt_r_axis[ax_lim[1]])

axes[2].plot(10 * np.log(td_bf_sum[:, max_resp_idx]))
axes[2].set_xticks(np.arange(td_bf_sum.shape[0])[::4], plt_trk_axis[::4])
axes[2].set_title(f"Max Response @ Cross-Track {plt_r_axis[max_resp_idx]:.2f}m")
axes[2].set_xlabel("Along-Track (m)")
axes[2].set_ylabel("Magnitude")

axes[3].plot(10 * np.log(td_bf_sum[td_bf_sum.shape[0]//2,...]))
axes[3].set_title("Cross-Track Result @ Center of Track")
axes[3].set_xticks(np.arange(td_bf_sum.shape[-1])[::50], plt_r_axis[::50])
axes[3].set_xlabel("Cross-Track (m)")
axes[3].set_ylabel("Magnitude")